In [ ]:
from dotenv import load_dotenv
from langsmith import traceable
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
load_dotenv()
os.environ["LANGCHAIN_PROJECT"] = "RAG-Based"


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite"
)


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## @traceable decorator is used to trace any python custom fucntion in langsmith
@traceable(name="load_documents")
def load_documents():

    loader = TextLoader("data.txt")

    documents = loader.load()

    return documents


@traceable(name="split_documents")
def split_documents(documents):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=50,
        chunk_overlap=10
    )

    chunks = splitter.split_documents(documents)

    return chunks


@traceable(name="create_vector_store")
def create_vector_store(chunks):

    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    return vectorstore


@traceable(name="retrieve_documents")
def retrieve_documents(vectorstore, query):

    retrieved_docs = vectorstore.similarity_search(
        query,
        k=3
    )

    return retrieved_docs


@traceable(name="build_context")
def build_context(retrieved_docs):

    context = ""

    for doc in retrieved_docs:

        context += doc.page_content
        context += "\n\n"

    return context


@traceable(name="generate_answer")
def generate_answer(context, query):

    prompt = f"""
    Answer the question ONLY from the given context.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    return response.content


@traceable(name="rag_pipeline")
def rag_pipeline(query):

    documents = load_documents()

    chunks = split_documents(documents)

    vectorstore = create_vector_store(chunks)

    retrieved_docs = retrieve_documents(vectorstore,query)

    context = build_context(retrieved_docs)

    answer = generate_answer(context,query)

    return answer


query = input("Ask Question: ")

answer = rag_pipeline(query)

print("\nAnswer:\n")
print(answer)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Answer:

LangSmith is used for tracing and monitoring LLM.
